# NHS England Prescribing — OpenPrescribing API & English Prescribing Dataset

This notebook demonstrates the **`nhs_prescribing`** accessor, which unifies two
complementary views of NHS England primary care (~7,000 GP practices) prescribing:

- **OpenPrescribing REST API** (EBM DataLab, University of Oxford) — monthly
  spending aggregates at national / CCG(SICBL) / practice level, prescribing
  quality measures (e.g. antibiotic stewardship), BNF & organisation code search
  and practice/CCG geolocations.
- **English Prescribing Dataset (EPD)** (NHS Business Services Authority) —
  practice-level bulk CSVs with SNOMED codes, ~600 million rows/year.

Typical uses: antimicrobial-resistance monitoring, treatment adherence,
health economics, and outbreak signal detection from prescription patterns.

No API key is required for either source; updates are monthly.

> **Note** — cells that query `openprescribing.net` or download EPD bulk files
> (multi-GB) are shown but were not executed in this copy. Everything else runs
> live against the NHSBSA open data portal.

In [1]:
from epidatasets.sources.nhs_prescribing import (
    NHSPrescribingAccessor,
    NHSPrescribingAPIError,
    bnf_to_flat,
    bnf_level,
    filter_bnf,
)

nhs = NHSPrescribingAccessor()  # optional: cache_dir=..., cache_ttl_days=7

print(nhs)
nhs.info()

NHS England primary care prescribing data: OpenPrescribing REST API (spending, prescribing quality measures, organisation locations) and the NHSBSA English Prescribing Dataset (EPD) monthly bulk CSVs. (https://openprescribing.net/api/)


{'name': 'nhs_prescribing',
 'description': 'NHS England primary care prescribing data: OpenPrescribing REST API (spending, prescribing quality measures, organisation locations) and the NHSBSA English Prescribing Dataset (EPD) monthly bulk CSVs.',
 'url': 'https://openprescribing.net/api/',
 'class': 'NHSPrescribingAccessor'}

## 1. Coverage

England (NHS primary care). The OpenPrescribing API covers monthly data from
2010; the EPD bulk files start in November 2020.

In [2]:
nhs.list_countries()

,country_code,country_name
0,GB-ENG,England


## 2. The BNF classification

Prescribing data is organised by the **British National Formulary** hierarchy.
Human-friendly *dotted* codes (`5.1.1.3` = broad-spectrum penicillins) map to
*flat* presentation codes (`0501013B0AAAAAA` = amoxicillin 500 mg capsules):

| Level | Dotted | Flat |
|---|---|---|
| Chapter — Infections | `5` | `05` |
| Section — Antibacterial drugs | `5.1` | `0501` |
| Paragraph — Penicillins | `5.1.1` | `050101` |
| Sub-paragraph — Broad-spectrum penicillins | `5.1.1.3` | `0501013` |
| Chemical — Amoxicillin | | `0501013B0` |
| Presentation — Amoxicillin 500 mg caps | | `0501013B0AAABAB` |

All accessor methods accept either form.

In [3]:
for dotted in ["5", "5.1", "5.1.1", "5.1.1.3"]:
    print(f"{dotted:9s} -> flat {bnf_to_flat(dotted):8s} ({bnf_level(dotted)})")

# flat codes pass straight through, chemicals/presentations included
print("0501013B0", "->", bnf_level("0501013B0"), "| 0501013B0AAABAB ->", bnf_level("0501013B0AAABAB"))

nhs.list_bnf_chapters()

5         -> flat 05       (chapter)
5.1       -> flat 0501     (section)
5.1.1     -> flat 050101   (paragraph)
5.1.1.3   -> flat 0501013  (subparagraph)
0501013B0 -> chemical | 0501013B0AAABAB -> presentation


,chapter_code,chapter_number,chapter_name,level
0,01,1,Gastro-Intestinal System,chapter
1,02,2,Cardiovascular System,chapter
2,03,3,Respiratory System,chapter
3,04,4,Central Nervous System,chapter
4,05,5,Infections,chapter
5,06,6,Endocrine System,chapter
6,07,7,"Obstetrics, Gynaecology and Urinary-Tract Diso...",chapter
7,08,8,Malignant Disease and Immunosuppression,chapter
8,09,9,Nutrition and Blood,chapter
9,10,10,Musculoskeletal and Joint Diseases,chapter


### Filtering any prescribing table by BNF prefix

In [4]:
import pandas as pd

sample = pd.DataFrame({"BNF_CODE": ["0501013B0AAABAB", "0501012H0AAAMAM",
                                    "0501055A0AAACAC", "0101021B0BEADAJ"]})
# keep only antibacterials (section 5.1), expressed in dotted form
filter_bnf(sample, "5.1")

,BNF_CODE
0,0501013B0AAABAB
1,0501012H0AAAMAM
2,0501055A0AAACAC


## 3. National spending (OpenPrescribing API)

Monthly All-England totals of items, quantity and actual cost for any BNF
code or prefix.

In [ ]:
spending = nhs.get_spending(bnf_code="5.1")  # antibacterial drugs
spending.tail(12)

In [ ]:
spending.plot(x="date", y="actual_cost", figsize=(12, 4),
             title="England — monthly spend on antibacterials (BNF 5.1)");

## 4. Spending by CCG and by practice

`year_month` accepts `"YYYY-MM"` or `"YYYYMM"`. Practice-level queries need an
organisation filter or a single month (upstream API constraint).

In [ ]:
ccg = nhs.get_spending_by_ccg(bnf_code="5.1", year_month="2024-01")
ccg.sort_values("actual_cost", ascending=False).head(10)

In [ ]:
practice = nhs.get_spending_by_practice(
    bnf_code="5.1",
    ccg_code="15N",       # all practices in this CCG...
    # practice_code="L81001",  # ...or a single practice
    year_month="2024-01",
)
practice.head(10)

## 5. Prescribing quality measures

OpenPrescribing computes ~150 quality measures — e.g. **antibiotic stewardship**
(`ktt9_antibiotics`), opioid monitoring, gabapentinoid prescribing. Without
`org_type` you get national series; with `org_type` you get organisation-level
percentiles.

In [ ]:
national = nhs.get_measures(measure="ktt9_antibiotics")
national[["id", "date", "numerator", "denominator", "calc_value"]].tail(12)

In [ ]:
practice_measure = nhs.get_measures(
    measure="ktt9_antibiotics",
    org_type="practice",
    org_code="L81001",
)
practice_measure.tail(12)

In [ ]:
# catalogue of all available measures, with plain-language rationale
definitions = nhs.get_measure_definitions()
definitions[["id", "name", "why_it_matters"]].head(10)

## 6. Organisation locations & geocoding

`org_location` returns GeoJSON; the accessor flattens it into a DataFrame with
`lat`/`lon` columns, ready for mapping.

In [ ]:
practices = nhs.get_practice_locations(ccg_code="15N")  # practices of a CCG
practices.head()

In [ ]:
ccgs = nhs.get_ccg_locations()  # CCG centroids
ccgs.head()

In [ ]:
# resolve specific org codes to coordinates
nhs.geocode_orgs(["L81001", "M85724"])

## 7. English Prescribing Dataset (EPD) — bulk files

The NHSBSA publishes one CSV per month (practice × presentation level,
SNOMED-coded, multi-GB each) on its open data portal. The accessor discovers
the files through the CKAN catalogue, streams them to a local cache and reads
them **chunked** so a full month never has to fit in memory.

In [5]:
months = nhs.list_epd_months()
print(f"{len(months)} monthly files available: {months[0]} .. {months[-1]}")

69 monthly files available: 202011 .. 202607


In [ ]:
# lazy, memory-bounded reading — filter to antibacterials chunk by chunk
chunks = nhs.get_epd_month(2024, 1, bnf_section="5.1", chunksize=500_000)

total_items = 0
for chunk in chunks:                 # streaming iterator
    total_items += chunk["ITEMS"].sum()

print(f"Antibacterial items dispensed in Jan 2024: {int(total_items):,}")

In [ ]:
# monthly time-series for a single chemical, e.g. amoxicillin (0501013B0)
ts = nhs.get_epd_timeseries("0501013B0", start_year_month="2024-01",
                            end_year_month="2024-06")
ts

In [ ]:
ts.plot(x="YEAR_MONTH", y="ITEMS", figsize=(12, 4),
        title="England — amoxicillin items dispensed (EPD)");

## 8. Using the plugin registry

In [6]:
from epidatasets import get_source, list_sources

print("nhs_prescribing" in list_sources())
acc = get_source("nhs_prescribing")
acc.list_countries()

True


,country_code,country_name
0,GB-ENG,England


## 9. Error handling & caching

Invalid parameters raise `ValueError`; network/HTTP failures are wrapped in
`NHSPrescribingAPIError` **with retry guidance** after automatic retries with
backoff. Responses and downloaded EPD files are cached on the filesystem
(7-day default TTL, aligned with the monthly release cycle).

In [7]:
try:
    nhs.get_spending_by_practice(bnf_code="5.1")  # missing org/month filter
except ValueError as e:
    print("ValueError:", e)

ValueError: Provide at least one of ccg_code, practice_code or year_month (the API does not return all practices over all months in a single request).


In [8]:
# live demonstration: API unreachable from this network -> wrapped error
try:
    nhs.get_spending(bnf_code="5.1", use_cache=False)
except NHSPrescribingAPIError as e:
    print("NHSPrescribingAPIError:", str(e)[:120], "...")

NHSPrescribingAPIError: HTTP 403 for https://openprescribing.net/api/1.0/spending/?code=0501&format=json. The service may be temporarily unavail ...


In [9]:
print("cache directory:", nhs.cache_dir)

# EPD files are large; a longer TTL avoids re-downloading them
nhs_long = NHSPrescribingAccessor(cache_ttl_days=30)
print("TTL:", nhs_long._cache_ttl)

cache directory: /home/fccoelho/.cache/epi_data/nhs_prescribing
TTL: 30 days, 0:00:00


## References

- OpenPrescribing: <https://openprescribing.net/> · API docs: <https://openprescribing.net/api/>
- NHSBSA English Prescribing Dataset: <https://www.nhsbsa.nhs.uk/prescription-data/prescribing-data/english-prescribing-data-epd>
- NHSBSA open data portal (EPD files): <https://opendata.nhsbsa.net>
- BNF classification: <https://www.bnf.org/>